# MA Bikeshare Coverage Analysis

Estimates the percentage of Massachusetts residents living within a **10-minute walk (~800m)** of a docked bikeshare station, for a Massachusetts DOT report.

### How to refresh or update

| If you want to... | Do this |
|---|---|
| Change the walking distance | Edit `BUFFER_METERS` in `config.py`, delete `data/processed/coverage_buffer.geojson`, re-run |
| Add/update station data | Drop new files in `data/raw/`, update paths in `config.py`, delete `data/processed/stations_combined.geojson`, re-run |
| Update Census data | Bump `ACS_YEAR` + `TIGER_URL` in `config.py`, delete `data/processed/ma_bg_geom_only.geojson` + `ma_bg_population.csv`, re-run |
| Include only year-round BlueBikes | Set `BLUEBIKES_SEASONAL_FILTER = 'Year Round'` in `config.py`, delete stations cache, re-run |
| Force full re-run from scratch | Delete all files in `data/processed/`, re-run |

**No notebook code needs to change — only `config.py`.**

---
## Section 1 — Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os, json, zipfile, io, tempfile
import pandas as pd, numpy as np
import geopandas as gpd, requests, openpyxl
from shapely.ops import unary_union
from keplergl import KeplerGl

sys.path.insert(0, '.')
from config import *

os.makedirs('data/processed', exist_ok=True)

print(f'Buffer distance:           {BUFFER_METERS}m')
print(f'ACS year:                  {ACS_YEAR}')
print(f'Population source:         {POPULATION_SOURCE}')
print(f'BlueBikes seasonal filter: {BLUEBIKES_SEASONAL_FILTER or "None (all stations)"}')

---
## Section 2 — Load Station Data

Reads from Excel files in `data/raw/`. On re-run, loads from the cached GeoJSON if it exists (fast path). Delete `data/processed/stations_combined.geojson` to force a refresh.

In [ ]:
def load_stations_from_excel():
    # BlueBikes — operator-published May 2026 station list (row 1 = metadata, row 2 = header)
    wb = openpyxl.load_workbook(BLUEBIKES_EXCEL, read_only=True)
    rows = list(wb.active.iter_rows(values_only=True)); wb.close()
    bb = []
    for r in rows[2:]:
        if not r[0] or r[2] is None or r[3] is None: continue
        if BLUEBIKES_SEASONAL_FILTER and r[4] != BLUEBIKES_SEASONAL_FILTER: continue
        bb.append({'station_name': r[1], 'latitude': float(r[2]),
                   'longitude': float(r[3]), 'municipality': r[5] or '', 'data_source': 'bluebikes'})

    # ValleyBike — matched rows from VB Crossreference tab (have BTS coordinates)
    wb = openpyxl.load_workbook(BTS_EXCEL, read_only=True)
    vbx = list(wb['VB Crossreference'].iter_rows(values_only=True)); wb.close()
    vb = []
    for row in vbx[3:]:
        if row[0] is None or str(row[0]).startswith('SECTION'): break
        pad = (row + (None,)*10)[:10]
        vb_name, hub_type, _, _, match_type, _, lat, lon, city, _ = pad
        if match_type in ('Non-Station', 'No BTS Match') or lat is None or lat == '—': continue
        try:
            vb.append({'station_name': str(vb_name).strip(), 'hub_type': hub_type,
                       'latitude': float(lat), 'longitude': float(lon),
                       'municipality': city or '', 'data_source': 'valleybike'})
        except (TypeError, ValueError): pass

    # Port Bikeshare — 5 primary docked stations in Newburyport (overflow racks excluded)
    wb = openpyxl.load_workbook(BTS_EXCEL, read_only=True)
    pr = list(wb['Port Bikeshare'].iter_rows(values_only=True)); wb.close()
    port = []
    for row in pr[3:]:
        if row[0] is None: continue
        pad = (row + (None,)*7)[:7]; nm, _, lat, lon, _, stype, _ = pad
        if stype and 'Overflow' in str(stype): continue
        if lat is None or lat == '—': continue
        try: port.append({'station_name': nm, 'latitude': float(lat), 'longitude': float(lon),
                          'municipality': 'Newburyport', 'data_source': 'port_bikeshare'})
        except (TypeError, ValueError): pass

    # MetroMobility — solar ChargeLock dock stations; 11 of 14 geocoded
    wb = openpyxl.load_workbook(BTS_EXCEL, read_only=True)
    mr = list(wb['Metromobility'].iter_rows(values_only=True)); wb.close()
    mm = []
    for row in mr[4:]:
        if row[0] is None: continue
        pad = (row + (None,)*8)[:8]; nm, _, city, lat, lon, _, _, use = pad
        if lat is None or lat == '—': continue
        if use and 'NO' in str(use): continue
        try: mm.append({'station_name': nm, 'latitude': float(lat), 'longitude': float(lon),
                        'municipality': city or '', 'data_source': 'metromobility'})
        except (TypeError, ValueError): pass

    KEEP = ['station_name', 'municipality', 'data_source', 'geometry']
    def to_gdf(data):
        return gpd.GeoDataFrame(pd.DataFrame(data),
            geometry=gpd.points_from_xy([r['longitude'] for r in data],
                                         [r['latitude']  for r in data]), crs=WGS84)
    return gpd.GeoDataFrame(
        pd.concat([to_gdf(src)[KEEP] for src in [bb, vb, port, mm]], ignore_index=True),
        geometry='geometry', crs=WGS84
    ).to_crs(MA_CRS)


if os.path.exists(CACHE_STATIONS):
    print(f'Loading cached stations...')
    all_stations = gpd.read_file(CACHE_STATIONS).to_crs(MA_CRS)
else:
    print('Reading stations from Excel files...')
    all_stations = load_stations_from_excel()
    all_stations.to_crs(WGS84).to_file(CACHE_STATIONS, driver='GeoJSON')

print('\nStation inventory:')
labels = {'bluebikes': 'BlueBikes', 'valleybike': 'ValleyBike',
          'port_bikeshare': 'Port Bikeshare', 'metromobility': 'MetroMobility'}
for k, v in all_stations['data_source'].value_counts().items():
    print(f'  {labels.get(k,k):<22} {v:>4}')
print(f'  {"TOTAL":<22} {len(all_stations):>4}')

---
## Section 3 — Load Census Data

**Geometries**: 2022 TIGER/Line cartographic boundary files for MA block groups (via Census TIGER).  
**Population**: ACS 5-year B01003 estimates via Census Reporter API (no API key required).  
Both are cached to `data/processed/` after first download — subsequent runs are instant.

To switch to the official Census API (requires free key from `api.census.gov/data/key_signup.html`): set `POPULATION_SOURCE = 'census_api'` and `CENSUS_API_KEY = 'your-key'` in `config.py`.

In [ ]:
GEOM_CACHE = 'data/processed/ma_bg_geom_only.geojson'
POP_CACHE  = 'data/processed/ma_bg_population.csv'

# ── Block group geometries ────────────────────────────────────────
if os.path.exists(GEOM_CACHE):
    print('Loading cached block group geometries...')
    bg_geom = gpd.read_file(GEOM_CACHE)
else:
    print('Downloading TIGER block groups (first run — will cache)...')
    resp = requests.get(TIGER_URL, timeout=180); resp.raise_for_status()
    with tempfile.TemporaryDirectory() as tmpdir:
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf: zf.extractall(tmpdir)
        shp_path = [os.path.join(tmpdir, f) for f in os.listdir(tmpdir) if f.endswith('.shp')][0]
        bg_geom = gpd.read_file(shp_path)
    bg_geom.to_crs(WGS84).to_file(GEOM_CACHE, driver='GeoJSON')
    print(f'  Saved: {GEOM_CACHE}')

# ── Population estimates ──────────────────────────────────────────
if os.path.exists(POP_CACHE):
    print('Loading cached ACS population...')
    pop_df = pd.read_csv(POP_CACHE)
else:
    if POPULATION_SOURCE == 'census_reporter':
        print('Fetching ACS population via Census Reporter (no key needed)...')
        resp = requests.get(CENSUS_REPORTER_URL, timeout=60); resp.raise_for_status()
        cr_data = resp.json()['data']
        pop_df = pd.DataFrame([
            {'GEOID': cr_geoid.replace('15000US', ''),
             'population': tables['B01003']['estimate']['B01003001']}
            for cr_geoid, tables in cr_data.items()
        ])
    else:  # census_api
        assert CENSUS_API_KEY, 'Set CENSUS_API_KEY in config.py (free at api.census.gov/data/key_signup.html)'
        print('Fetching ACS population via Census API...')
        url = ACS_API_URL.format(year=ACS_YEAR, key=CENSUS_API_KEY)
        resp = requests.get(url, timeout=90); resp.raise_for_status()
        acs_json = resp.json()
        acs_df = pd.DataFrame(acs_json[1:], columns=acs_json[0])
        pop_df = pd.DataFrame({
            'GEOID': acs_df['state'] + acs_df['county'] + acs_df['tract'] + acs_df['block group'],
            'population': pd.to_numeric(acs_df['B01003_001E'], errors='coerce').fillna(0)
        })
    pop_df.to_csv(POP_CACHE, index=False)
    print(f'  Saved: {POP_CACHE}')

# ── Join + project ────────────────────────────────────────────────
block_groups = bg_geom.merge(pop_df, on='GEOID', how='left').to_crs(MA_CRS)
block_groups['population']    = block_groups['population'].fillna(0)
block_groups['total_area_m2'] = block_groups.geometry.area
TOTAL_MA_POP = block_groups['population'].sum()

print(f'\nBlock groups: {len(block_groups):,}  |  Total MA population: {TOTAL_MA_POP:,.0f}')

---
## Section 4 — 800m Buffer + Population Coverage

Draws an 800m Euclidean circle around each station, unions all circles, then applies **areal interpolation** to estimate the resident population within the coverage zone.

```
covered_pop(block_group) = population × (overlap_area / total_area)
```

In [ ]:
# ── Buffer ────────────────────────────────────────────────────────
if os.path.exists(CACHE_BUFFER):
    print('Loading cached buffer polygon...')
    coverage_gdf = gpd.read_file(CACHE_BUFFER).to_crs(MA_CRS)
else:
    print(f'Computing {BUFFER_METERS}m buffers...')
    coverage_union = unary_union(all_stations.geometry.buffer(BUFFER_METERS))
    coverage_gdf   = gpd.GeoDataFrame(geometry=[coverage_union], crs=MA_CRS)
    coverage_gdf.to_crs(WGS84).to_file(CACHE_BUFFER, driver='GeoJSON')

area_km2 = coverage_gdf.to_crs(MA_CRS).geometry.area.sum() / 1_000_000
print(f'Coverage area: {area_km2:.1f} km²  ({area_km2/27336*100:.1f}% of MA land area)')

# ── Areal interpolation ───────────────────────────────────────────
if os.path.exists(CACHE_BG_COVERAGE):
    print('Loading cached areal interpolation results...')
    bg = gpd.read_file(CACHE_BG_COVERAGE).to_crs(MA_CRS)
else:
    print('Running areal interpolation...')
    bg_proj    = block_groups.to_crs(MA_CRS)
    cov_proj   = coverage_gdf.to_crs(MA_CRS)
    covered    = gpd.overlay(bg_proj, cov_proj[['geometry']], how='intersection')
    covered['covered_area_m2'] = covered.geometry.area
    cov_area   = covered.groupby('GEOID')['covered_area_m2'].sum().reset_index()
    bg         = block_groups.merge(cov_area, on='GEOID', how='left')
    bg['covered_area_m2']    = bg['covered_area_m2'].fillna(0)
    bg['total_area_m2']      = bg.geometry.area
    bg['coverage_fraction']  = (bg['covered_area_m2'] / bg['total_area_m2']).clip(0, 1)
    bg['covered_population'] = bg['population'] * bg['coverage_fraction']
    bg['pop_density_km2']    = bg['population'] / (bg['total_area_m2'] / 1_000_000).replace(0, np.nan)
    bg.to_crs(WGS84).to_file(CACHE_BG_COVERAGE, driver='GeoJSON')

covered_pop = bg['covered_population'].sum()
total_pop   = bg['population'].sum()
pct         = (covered_pop / total_pop) * 100

print(f'\n{"="*52}')
print(f'  {pct:.1f}% of MA residents ({covered_pop:,.0f} people)')
print(f'  live within {BUFFER_METERS}m of a docked bikeshare station.')
print(f'{"="*52}')

---
## Section 5 — KeplerGL Map

Three layers:
- **Block Groups** — choropleth by population density (people/km²) — YlOrRd color scale
- **Coverage (800m Buffer)** — union polygon, semi-transparent cyan with bright stroke
- **Bikeshare Stations** — dots colored by system: blue = BlueBikes, green = ValleyBike, orange = Port, purple = MetroMobility

In [ ]:
# Simplify block group geometry for faster rendering (0.0005° ≈ 40m tolerance)
bg_vis = bg[['GEOID','population','coverage_fraction','pop_density_km2','geometry']].copy()
bg_vis['pop_density_km2'] = bg_vis['pop_density_km2'].fillna(0).round(0).astype(int)
bg_vis['coverage_pct']    = (bg_vis['coverage_fraction'].fillna(0) * 100).round(1)
bg_vis['geometry']        = bg_vis['geometry'].simplify(0.0005, preserve_topology=True)
bg_vis = bg_vis[bg_vis['geometry'].notna() & ~bg_vis['geometry'].is_empty].to_crs(WGS84)

buf_raw    = gpd.read_file(CACHE_BUFFER)
buffer_vis = gpd.GeoDataFrame(
    geometry=[buf_raw.geometry.unary_union.simplify(0.0002, preserve_topology=True)], crs=buf_raw.crs
)

stations_flat = pd.DataFrame({
    'station_name': all_stations['station_name'],
    'data_source':  all_stations['data_source'],
    'municipality': all_stations['municipality'],
    'latitude':     all_stations.to_crs(WGS84).geometry.y,
    'longitude':    all_stations.to_crs(WGS84).geometry.x,
})

# keplergl serializes GeoDataFrame geometry as WKT in a column named "geometry".
# The geojson layer must reference it via "columns": {"geojson": "geometry"}.
# Basemap: Carto Dark Matter — free, no Mapbox token needed.
kepler_config = {
    "version": "v1",
    "config": {
        "visState": {
            "filters": [],
            "layers": [
                {"id": "bg", "type": "geojson",
                 "config": {"dataId": "Block Groups", "label": "Population Density",
                             "isVisible": True,
                             "columns": {"geojson": "geometry"},
                             "visConfig": {
                                 "opacity": 0.75, "strokeOpacity": 0.2, "thickness": 0.3,
                                 "strokeColor": [50, 50, 50],
                                 "colorRange": {"name": "ColorBrewer YlOrRd-6",
                                                "type": "sequential", "category": "ColorBrewer",
                                                "colors": ["#ffffb2","#fed976","#feb24c","#fd8d3c","#f03b20","#bd0026"]},
                                 "stroked": True, "filled": True}},
                 "visualChannels": {"colorField": {"name": "pop_density_km2", "type": "integer"},
                                    "colorScale": "quantile"}},
                {"id": "buf", "type": "geojson",
                 "config": {"dataId": "Coverage (800m Buffer)", "label": "Coverage (800m Buffer)",
                             "color": [0, 210, 230], "isVisible": True,
                             "columns": {"geojson": "geometry"},
                             "visConfig": {
                                 "opacity": 0.22, "strokeOpacity": 1.0, "thickness": 2.5,
                                 "strokeColor": [0, 240, 255],
                                 "stroked": True, "filled": True}},
                 "visualChannels": {"colorField": None}},
                {"id": "stations", "type": "point",
                 "config": {"dataId": "Bikeshare Stations", "label": "Bikeshare Stations",
                             "columns": {"lat": "latitude", "lng": "longitude", "altitude": None},
                             "isVisible": True,
                             "visConfig": {
                                 "radius": 5, "opacity": 0.95, "outline": True, "thickness": 0.8,
                                 "strokeColor": [240, 240, 240],
                                 "colorRange": {"name": "Custom", "type": "qualitative",
                                                "category": "Custom",
                                                "colors": ["#29B6F6","#66BB6A","#FFA726","#AB47BC"],
                                                "colorMap": [["bluebikes","#29B6F6"],["valleybike","#66BB6A"],
                                                             ["port_bikeshare","#FFA726"],["metromobility","#AB47BC"]]},
                                 "filled": True}},
                 "visualChannels": {"colorField": {"name": "data_source", "type": "string"},
                                    "colorScale": "ordinal"}}
            ],
            "interactionConfig": {
                "tooltip": {
                    "fieldsToShow": {
                        "Bikeshare Stations": [{"name":"station_name"},{"name":"data_source"},{"name":"municipality"}],
                        "Coverage (800m Buffer)": [],
                        "Block Groups": [{"name":"population"},{"name":"coverage_pct"},{"name":"pop_density_km2"}]
                    },
                    "enabled": True
                },
                "brush": {"size": 0.5, "enabled": False}
            },
            "layerBlending": "normal"
        },
        "mapState": {"bearing": 0, "dragRotate": False,
                     "latitude": 42.30, "longitude": -71.55,
                     "pitch": 0, "zoom": 8.5, "isSplit": False},
        "mapStyle": {
            "styleType": "carto_dark",
            "mapStylesReplaceDefault": True,
            "mapStyles": {
                "carto_dark": {
                    "id": "carto_dark",
                    "label": "Dark (Carto)",
                    "url": "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
                    "icon": "https://carto.com/help/images/building-maps/basemaps/dark_nolabels.png",
                    "layerGroups": []
                }
            },
            "visibleLayerGroups": {}
        }
    }
}

m = KeplerGl(height=KEPLER_HEIGHT, config=kepler_config)
m.add_data(data=bg_vis,        name='Block Groups')
m.add_data(data=buffer_vis,    name='Coverage (800m Buffer)')
m.add_data(data=stations_flat, name='Bikeshare Stations')
m.save_to_html(file_name=OUT_MAP_HTML, center_map=False)
print(f'Standalone map saved to: {OUT_MAP_HTML}')
m

---
## Section 6 — Results Summary

In [ ]:
print('STATION INVENTORY')
print(f'{"System":<25} {"Stations":>8}')
print('-' * 35)
labels = {'bluebikes': 'BlueBikes', 'valleybike': 'ValleyBike',
          'port_bikeshare': 'Port Bikeshare', 'metromobility': 'MetroMobility'}
for system, n in all_stations['data_source'].value_counts().items():
    print(f'{labels.get(system, system):<25} {n:>8}')
print(f'{"TOTAL":<25} {len(all_stations):>8}')

print()
print('COVERAGE RESULT  (800m Euclidean Buffer)')
print(f'  Total MA population:     {total_pop:>14,.0f}')
print(f'  Population within 800m:  {covered_pop:>14,.0f}')
print(f'  Coverage:                {pct:>13.1f}%')
print(f'  Coverage area:           {area_km2:>12.0f} km²')

print()
print('CAVEATS')
print('  - Euclidean buffer overestimates vs. true street-network walkshed')
print('  - ValleyBike Drop Hubs (geofenced zones) are included — see limitations')
print('  - 3 MetroMobility cities unlocated (Medford, Quincy, Mattapan)')
print('  - Port Bikeshare: 4 of 5 coordinates are approximate')
print(f'  - Population: ACS {ACS_YEAR} 5-year estimates')

---
## Section 7 — Network Walkshed (not yet active)

When ready, this section computes true 10-minute pedestrian isochrones via OSMnx, producing a more accurate (smaller) coverage estimate to compare against the buffer above.

**To activate**: uncomment the code below. First download takes 10–15 min to cache the MA walking network (~500MB). Subsequent runs use the cache.

In [ ]:
# import osmnx as ox, networkx as nx
# from shapely.geometry import MultiPoint
# ox.settings.use_cache = True; ox.settings.log_console = False
#
# WALK_SECONDS = WALK_MINUTES * 60
# print('Downloading MA walking network...')
# G = ox.graph_from_bbox(north=42.886, south=41.237, east=-69.928, west=-73.508,
#                        network_type='walk', simplify=True)
# G = ox.add_edge_speeds(G, fallback=WALK_SPEED_KPH * 1000 / 3600)
# G = ox.add_edge_travel_times(G)
#
# def station_isochrone(G, lat, lon, t):
#     try:
#         c = ox.nearest_nodes(G, lon, lat)
#         sub = nx.ego_graph(G, c, radius=t, distance='travel_time')
#         pts = [Point(d['x'], d['y']) for _, d in sub.nodes(data=True)]
#         return MultiPoint(pts).convex_hull if len(pts) >= 3 else None
#     except Exception: return None
#
# s84 = all_stations.to_crs(WGS84)
# isos = [station_isochrone(G, r.geometry.y, r.geometry.x, WALK_SECONDS) for _, r in s84.iterrows()]
# walkshed = gpd.GeoDataFrame(geometry=[unary_union([p for p in isos if p])], crs=WGS84).to_crs(MA_CRS)
# walkshed.to_file('data/processed/coverage_walkshed.geojson', driver='GeoJSON')
#
# # Areal interpolation for walkshed
# cov_ws    = gpd.overlay(block_groups, walkshed[['geometry']], how='intersection')
# cov_ws['covered_area_m2'] = cov_ws.geometry.area
# bg_ws = block_groups.merge(cov_ws.groupby('GEOID')['covered_area_m2'].sum().reset_index(), on='GEOID', how='left')
# bg_ws['coverage_fraction'] = (bg_ws['covered_area_m2'].fillna(0) / bg_ws['total_area_m2']).clip(0,1)
# ws_pop = (bg_ws['population'] * bg_ws['coverage_fraction']).sum()
# ws_pct = ws_pop / TOTAL_MA_POP * 100
# print(f'Network walkshed: {ws_pct:.1f}%  vs  buffer: {pct:.1f}%  (diff: {pct-ws_pct:.1f}pp)')
print('Network walkshed: not yet active.')